In [1]:

import pandas as pd
#get the data
path = "../data/processed/MBP10_ZC.parquet"
df = pd.read_parquet(path, columns=["ts_event", "bid_px_05", "ask_px_05"])
df = df.sort_values("ts_event").reset_index(drop=True)

step = 5 
df 

,ts_event,bid_px_05,ask_px_05
0,2025-02-16 13:00:06.432206309+00:00,495.5,497.00
1,2025-02-16 22:00:00.337137947+00:00,495.5,497.00
2,2025-02-16 22:00:22.296140633+00:00,495.5,497.00
3,2025-02-16 22:00:22.538433281+00:00,495.5,497.00
4,2025-02-16 22:00:23.483635199+00:00,495.5,497.00
...,...,...,...
30839149,2026-02-15 22:00:10.648420233+00:00,431.0,431.75
30839150,2026-02-15 22:00:11.228268619+00:00,431.0,431.75
30839151,2026-02-15 22:00:12.104053681+00:00,431.0,431.75
30839152,2026-02-15 22:00:12.716769795+00:00,431.0,431.50


In [ ]:

# downsample 
df_plot = (
    df.set_index("ts_event")
    .resample("5min")[["bid_px_05", "ask_px_05"]]
    .last()
    .dropna()
    .reset_index()
)

In [3]:
import plotly.graph_objects as go

# plot
y_pad = (df_plot["ask_px_05"].max() - df_plot["bid_px_05"].min()) * 0.05

fig = go.Figure()

# Ask first — so fill direction goes ask -> bid
fig.add_trace(go.Scatter(
    x=df_plot["ts_event"], y=df_plot["ask_px_05"],
    name="Ask",
    line=dict(color="#EF5350", width=1.8),
    mode="lines"
))

# Bid — fills tonexty to shade the spread band
fig.add_trace(go.Scatter(
    x=df_plot["ts_event"], y=df_plot["bid_px_05"],
    name="Bid",
    line=dict(color="#26A65B", width=1.8),
    fill="tonexty",
    fillcolor="rgba(160, 160, 160, 0.15)",
    mode="lines"
))

fig.update_layout(
    title=dict(
        text="Bid/Ask Price (Level 5) — ZC MBP10<br><span style='font-size:15px;font-weight:normal;'>1-Year | 5-min resampled | Spread shaded</span>",
        x=0.5, xanchor="center"
    ),
    legend=dict(orientation="h", yanchor="top", y=-0.12, xanchor="center", x=0.5),
    hovermode="x unified",
    yaxis=dict(
        range=[df_plot["bid_px_05"].min() - y_pad, df_plot["ask_px_05"].max() + y_pad],
        showgrid=True, gridcolor="rgba(128,128,128,0.12)", zeroline=False,
    ),
    xaxis=dict(
        showgrid=True, gridcolor="rgba(128,128,128,0.12)",
        dtick="M1", tickformat="%b '%y",
    ),
    margin=dict(l=70, r=40, t=110, b=80),
)

fig.update_yaxes(title_text="Price (USD/bu)")
fig.show()


# Results (as a chart)
![Image of level 5 bid ask of corn](../outputs/charts/bid_ask_chart_ZC.png)

